# Simulación nula (pedida por la Dra. Gaur): ¿el análisis inventa un techo de capacidad adaptativa?
Se generan **pacientes falsos** donde la verdad es conocida y se les aplica el análisis del protocolo: recuperación tras una perturbación (BSI en T0+W menos el nadir), recuperación esperada con validación cruzada por paciente, regresión por cuantiles y contraste de pendientes **τ=0.90 menos τ=0.50**.

- **Mundo A:** NO hay techo. Si el análisis "detecta" techo aquí, es un **falso techo**.
- **Mundo B:** SÍ hay techo (peor estado basal → menor fracción recuperable). Sirve para medir potencia y cuántos pacientes hacen falta.
- Tres resultados, reportados todos y sin elegir después de ver: `AC_R` (recuperación en bruto), `AC_frac` (fracción de la caída recuperada), `END_given_nadir` (BSI final dado basal y nadir).

**Reglas fijadas antes de correr** (están también en `RULES` del código):
1. Regla cruda aceptable solo si falsos techos ≤ 0.07.
2. Si no, la detección se calibra contra el nulo simulado (estimado > percentil 95 del nulo).
3. Suficiencia: menor n con potencia calibrada ≥ 0.80.
4. Respaldo: τ=0.75 como primario o PE3 exploratorio.

**Límites:** el modelo generador (ruido, muestreo, acoplamiento, una perturbación por paciente) son **supuestos míos** hasta tener el diccionario del hospital; los números definitivos se recalibran con ruido y frecuencia reales. Tarda ~20-25 min con la configuración completa (150 repeticiones; en la corrida registrada abajo fueron 1295 s).

In [1]:
%%writefile nullsim.py
"""Simulacion nula (Dra. Gaur): ¿el pipeline 'recuperacion tras perturbacion + regresion por cuantiles'
inventa un techo de capacidad adaptativa cuando NO existe?  Pacientes falsos; la verdad es conocida.
REGLAS FIJADAS ANTES DE CORRER (ver RULES). BSI: mayor = mejor estado (escala 0-1)."""
import warnings
import numpy as np, pandas as pd
from statsmodels.regression.quantile_regression import QuantReg
from statsmodels.tools.sm_exceptions import IterationLimitWarning
warnings.simplefilter('ignore', IterationLimitWarning); warnings.filterwarnings('ignore')

SEED = 2026
W = 30.0            # ventana fija por tipo de perturbacion (infeccion aguda); NO depende de la trayectoria
ALPHA_ONE_SIDED = 0.05
RULES = ("PRUEBA PRIMARIA: contraste de pendientes (cuantil 0.90 menos cuantil 0.50) de la capacidad adaptativa AC sobre el BSI basal, "
         "ajustando por edad. 'Techo detectado' = limite inferior del intervalo bootstrap por paciente (percentil 5, una cola) del contraste > 0. "
         "FALSO TECHO = detectado en el mundo A (sin techo). Se considera ACEPTABLE si la tasa de falsos techos <= 0.07. "
         "POTENCIA objetivo >= 0.80 en el mundo B. Suficiencia: menor n con potencia >= 0.80. "
         "Respaldo si no se alcanza: tau 0.75 como primario, o PE3 exploratorio. "
         "Si la regla cruda supera 0.07 de falsos techos, NO es aceptable tal cual: la deteccion se calibra contra el nulo simulado "
         "(techo detectado = estimado por encima del percentil 95 del nulo con el mismo ruido y muestreo). "
         "Se reportan los 3 resultados (AC_R = recuperacion en bruto, AC_frac = fraccion de la caida recuperada, END_given_nadir = BSI final dado basal y nadir) sin elegir el mejor despues de ver.")

sig = lambda x: 1 / (1 + np.exp(-x))
logit = lambda p: np.log(p / (1 - p))

def simulate(n, world, noise, dens_sev, kappa=0.5, coupling=0.25, rng=None):
    """world 'A' = sin techo; 'B' = techo: peor estado basal -> menor fraccion maxima recuperable.
    noise: sd del ruido de medicion (escala logit). dens_sev: >0 = mas mediciones en los mas graves.
    coupling: cuanto mas alto el BSI basal, mas puede caer (acoplamiento matematico natural, presente en A y B)."""
    rng = rng or np.random.default_rng()
    z_age = rng.normal(size=n); age = 60 + 10 * z_age
    theta = -0.3 * z_age + np.sqrt(1 - 0.09) * rng.normal(size=n)   # estado basal latente (mayor = mejor)
    S = rng.uniform(0, 1, n)                                          # gravedad, independiente del BSI
    x0 = 0.5 + theta
    D = (0.8 + 1.2 * S) * np.exp(0.15 * rng.normal(size=n)) * (1 + coupling * theta).clip(0.3)
    base_frac = rng.beta(5, 2, n)
    u = np.ones(n) if world == "A" else 1 - kappa * (1 - sig(1.5 * theta))
    rho = u * base_frac
    tau = 6 * np.exp(0.5 * S + 0.2 * rng.normal(size=n))
    def true_x(i, t):
        t = np.asarray(t, float); deficit = np.where(t < 0, 0.0, D[i] * (1 - np.exp(-t / 1.0)) * ((1 - rho[i]) + rho[i] * np.exp(-t / tau[i])))
        return x0[i] - deficit
    rows = []
    for i in range(n):
        n_pre = 1 + rng.poisson(1.0); t_pre = rng.uniform(-90, -1, n_pre)
        lam = 3.0 + dens_sev * S[i]; n_post = rng.poisson(lam); t_post = np.sort(rng.uniform(0.5, W, n_post))
        t = np.concatenate([t_pre, t_post]); x = true_x(i, t)
        bsi = sig(x + noise * rng.normal(size=len(t)))
        for tt, b in zip(t, bsi): rows.append((i, tt, b))
    obs = pd.DataFrame(rows, columns=["id", "t", "bsi"])
    pat = pd.DataFrame({"id": np.arange(n), "age": age, "S": S})
    return obs, pat

def patient_table(obs, pat):
    """Una fila por paciente con BSI basal, nadir, final de ventana y resultados. Exige >=1 medicion previa y >=2 posteriores
    (esta restriccion SELECCIONA pacientes; se reporta cuantos quedan)."""
    pre = obs[obs.t < 0].groupby("id").bsi.agg(bsi0="mean", n_pre="size")
    post = obs[obs.t >= 0].sort_values(["id", "t"]).groupby("id").agg(nadir=("bsi", "min"), end=("bsi", "last"), n_post=("bsi", "size"))
    d = pat.set_index("id").join(pre).join(post).dropna()
    d = d[(d.n_post >= 2) & (d.n_pre >= 1)].copy()
    d["R"] = d["end"] - d["nadir"]
    d["drop"] = d["bsi0"] - d["nadir"]
    d["frac"] = np.where(d["drop"] > 0.05, d["R"] / d["drop"].where(d["drop"] > 0.05), np.nan)
    return d.reset_index()

def crossfit_resid(d, y, K=5, seed=0):
    """Recuperacion esperada con validacion cruzada por paciente (SIN BSI basal); AC = observado - esperado."""
    rng = np.random.default_rng(seed); n = len(d); idx = rng.permutation(n); folds = np.array_split(idx, K)
    X = np.column_stack([np.ones(n), d["S"].values, d["age"].values]); e = np.zeros(n)
    for f in folds:
        tr = np.setdiff1d(idx, f); b = np.linalg.lstsq(X[tr], y[tr], rcond=None)[0]; e[f] = X[f] @ b
    return y - e

def qslope(y, X, tau):
    # variables estandarizadas (mejor convergencia); la pendiente queda en unidades de DE de y por DE de BSI basal
    ys = (y - y.mean()) / y.std(); Xs = X.copy()
    for j in range(1, X.shape[1]): Xs[:, j] = (X[:, j] - X[:, j].mean()) / X[:, j].std()
    return QuantReg(ys, Xs).fit(q=tau, max_iter=1000, p_tol=1e-6).params[1]

def contrast(y, b0, age, taus=(0.90, 0.50), extra=None):
    cols = [np.ones(len(y)), b0, age] + ([] if extra is None else [extra])
    X = np.column_stack(cols)
    return qslope(y, X, taus[0]) - qslope(y, X, taus[1])

def test_contrast(d, outcome, B=100, rng=None, taus=(0.90, 0.50)):
    rng = rng or np.random.default_rng()
    dd = d.dropna(subset=["bsi0", "age", "S"] + ([outcome] if outcome in d else []))
    if outcome == "AC_R":
        dd = dd.copy(); dd["y"] = crossfit_resid(dd, dd["R"].values); extra = None
    elif outcome == "AC_frac":
        dd = dd.dropna(subset=["frac"]).copy(); dd["y"] = crossfit_resid(dd, dd["frac"].values); extra = None
    elif outcome == "END_given_nadir":
        dd = dd.copy(); dd["y"] = dd["end"].values; extra = "nadir"
    else: raise ValueError(outcome)
    y, b0, age = dd["y"].values, dd["bsi0"].values, dd["age"].values
    ex = dd[extra].values if extra else None
    est = contrast(y, b0, age, taus, ex); n = len(dd); bs = np.empty(B)
    for k in range(B):
        i = rng.integers(0, n, n)
        bs[k] = contrast(y[i], b0[i], age[i], taus, None if ex is None else ex[i])
    lo, hi = np.percentile(bs, [100 * ALPHA_ONE_SIDED, 100 - 100 * ALPHA_ONE_SIDED])
    return {"n": n, "est": est, "lo": lo, "hi": hi, "width": hi - lo, "detect": bool(lo > 0)}

OUTS = ["AC_R", "AC_frac", "END_given_nadir"]

def est_only(d, outcome):
    dd = d.dropna(subset=["bsi0", "age", "S"]).copy(); extra = None
    if outcome == "AC_R": dd["y"] = crossfit_resid(dd, dd["R"].values)
    elif outcome == "AC_frac": dd = dd.dropna(subset=["frac"]).copy(); dd["y"] = crossfit_resid(dd, dd["frac"].values)
    else: dd["y"] = dd["end"].values; extra = dd["nadir"].values
    return contrast(dd["y"].values, dd["bsi0"].values, dd["age"].values, (0.90, 0.50), extra)

def one_rep(args):
    n, world, noise, dens, kappa, coupling, seed, do_boot, B = args
    rng = np.random.default_rng(seed)
    obs, pat = simulate(n, world, noise, dens, kappa, coupling=coupling, rng=rng); d = patient_table(obs, pat)
    out = {"world": world, "noise": noise, "dens": dens, "n": n, "kappa": kappa, "coupling": coupling, "kept": len(d) / n}
    for o in OUTS: out["est_" + o] = est_only(d, o)
    out["detect_AC_R"] = test_contrast(d, "AC_R", B=B, rng=rng)["detect"] if (do_boot and world == "A") else np.nan
    return out

def run_grid(scenarios, reps, boot_reps, B=50, n_jobs=-1):
    """scenarios: lista de (n, world, noise, dens, kappa). boot_reps = cuantas repeticiones llevan bootstrap (regla cruda)."""
    from joblib import Parallel, delayed
    jobs = []
    for si, sc in enumerate(scenarios):
        n, w, nz, dn, kp = sc[:5]; cp = sc[5] if len(sc) > 5 else 0.25
        for r in range(reps): jobs.append((n, w, nz, dn, kp, cp, SEED + 1000 * si + r, r < boot_reps, B))
    return pd.DataFrame(Parallel(n_jobs=n_jobs)(delayed(one_rep)(j) for j in jobs))

def summarize(res):
    rows = []
    key = ["n", "noise", "dens", "coupling"]
    for k, g in res.groupby(key):
        A = g[g.world == "A"]
        if A.empty: continue
        for kap, Bg in g[g.world == "B"].groupby("kappa"):
            row = dict(zip(key, k)); row["kappa"] = kap; row["kept_%"] = round(100 * g.kept.mean())
            row["falso_techo_regla_cruda_AC_R"] = round(A.detect_AC_R.dropna().astype(float).mean(), 2) if A.detect_AC_R.notna().any() else np.nan
            for o in OUTS:
                q95 = np.percentile(A["est_" + o], 95)
                row["est_A_" + o] = round(A["est_" + o].mean(), 3); row["est_B_" + o] = round(Bg["est_" + o].mean(), 3)
                row["potencia_calibrada_" + o] = round((Bg["est_" + o] > q95).mean(), 2)
            rows.append(row)
    return pd.DataFrame(rows)


Writing nullsim.py


In [2]:
# Corre toda la simulacion (no hay que subir nada). Para una prueba rapida cambia REPS=30, BOOT=10.
import pandas as pd, time, importlib, nullsim as N
importlib.reload(N)
REPS, BOOT = 150, 40
sc = []
for nz in (0.3, 0.6):
    for dn in (0.0, 4.0):
        sc += [(500, "A", nz, dn, 0.5), (500, "B", nz, dn, 0.5)]
for n in (200, 500, 1000, 2000):
    sc += [(n, "A", 0.3, 4.0, 0.5), (n, "B", 0.3, 4.0, 0.5), (n, "B", 0.3, 4.0, 0.3)]
sc += [(1000, "A", 0.3, 4.0, 0.5, 0.0), (1000, "B", 0.3, 4.0, 0.5, 0.0)]   # control: SIN acoplamiento
t = time.time(); res = N.run_grid(sc, reps=REPS, boot_reps=BOOT, B=40); print('listo en', round(time.time() - t), 's')
res.to_csv('resultados_simulacion.csv', index=False)

listo en 1295 s


In [3]:
# Resultado principal
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
sm = N.summarize(res)
cols = ['n','noise','dens','coupling','kappa','kept_%','falso_techo_regla_cruda_AC_R','potencia_calibrada_AC_R','potencia_calibrada_AC_frac','potencia_calibrada_END_given_nadir']
print(sm[cols].to_string(index=False))
print('\nfalso_techo_regla_cruda_AC_R = fraccion de veces que el analisis del protocolo detecta techo en el mundo SIN techo (objetivo <= 0.07).')
print('potencia_calibrada_* = fraccion de veces que detecta el techo real (objetivo >= 0.80) usando el nulo simulado como referencia.')

   n  noise  dens  coupling  kappa  kept_%  falso_techo_regla_cruda_AC_R  potencia_calibrada_AC_R  potencia_calibrada_AC_frac  potencia_calibrada_END_given_nadir
 200    0.3   4.0      0.25    0.3      94                          0.22                     0.15                        0.13                                0.07
 200    0.3   4.0      0.25    0.5      94                          0.22                     0.15                        0.19                                0.13
 500    0.3   0.0      0.25    0.5      80                          0.50                     0.27                        0.49                                0.26
 500    0.3   4.0      0.25    0.3      94                          0.50                     0.17                        0.22                                0.13
 500    0.3   4.0      0.25    0.5      94                          0.50                     0.25                        0.45                                0.21
 500    0.6   0.0      0.25 